# XGBoost регрессия

Один эксперимент без сохранения результатов.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer


def cv(model, X, y):
    scores = cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring='neg_mean_absolute_percentage_error',
        n_jobs=1,
    )
    mape = -scores
    print(f"MAPE: {mape.mean():.6f} +- {mape.std():.6f}")
    return mape.mean()


df_train = pd.read_csv('data/train.csv')
X = df_train.iloc[:, :-1].copy()
y = df_train.iloc[:, -1].copy()

X[['unified_address_city', 'unified_address_region']] = X[['unified_address_city', 'unified_address_region']].fillna('missing')
for col in ['key_skills_name', 'languages_name', 'employer_industries']:
    if col in X.columns:
        X[col] = X[col].fillna('missing')

X = X.drop(columns=['id','employer_id','raw_description','raw_branded_description','lemmaized_wo_stopwords_raw_description','lemmaized_wo_stopwords_raw_branded_description','name','unified_address_country'], errors='ignore').copy()

counts = X['employer_name'].value_counts()
rare_categories = counts[counts < 200].index
X['employer_name'] = X['employer_name'].replace(rare_categories, 'Other')

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

pre = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc', StandardScaler()),
    ]), num_cols),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ]), cat_cols),
])

from xgboost import XGBRegressor


## Эксперимент

In [2]:
model = Pipeline([
    ('preprocessor', pre),
    ('model', XGBRegressor(
        objective='reg:absoluteerror',
        n_estimators=1200,
        learning_rate=0.03,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=2.0,
        n_jobs=1,
        tree_method='hist',
    )),
])

res = cv(model, X, y)
res


MAPE: 0.366228 +- 0.017745


np.float64(0.36622801489788015)